# Challenge 2: Versioned dataset and first eval

An **offline eval** runs your agent over a fixed dataset and scores every output. Three pieces:

1. **Dataset:** inputs plus reference outputs. You'll pin it to a **version** so every later run scores the same questions.
2. **Target function:** runs your agent on one example's inputs.
3. **Evaluators:** score the target's output against the reference.

**One cell was blank in the workshop. Here it is filled in, so every cell runs.**

In [ ]:
import os
import sys

for path in ("..", "../lab"):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from langsmith import Client, uuid7

import lab_eval
from eval_lab.agent import SYSTEM_PROMPT_V1, TOOLS, build_agent, extract_sources, extract_tool_calls

client = Client()
DATASET = lab_eval.dataset_name()
print("project:", os.environ["LANGSMITH_PROJECT"])
print("dataset:", DATASET)

## The dataset

Eight questions of the kind employees ask. Each has an `inputs.query` and reference `outputs`:

| Reference field | Used for |
|---|---|
| `keywords` | A cheap string check on the answer (this challenge) |
| `trajectory` | The tools the agent should use (challenge 3) |
| `reference_answer` | A note for humans reviewing rows in LangSmith |

Example 8 asks about something the knowledge base does **not** cover. The right behavior is to say so, not to invent a policy.

In [ ]:
EXAMPLES = [
    {"inputs": {"query": "How quickly must a Sev1 incident be escalated to on-call?"},
     "outputs": {"reference_answer": "Says Sev1 incidents must be escalated within 15 minutes (KB-101).",
                 "keywords": ["15"], "trajectory": ["search_confluence"]}},
    {"inputs": {"query": "How long do we retain transaction records?"},
     "outputs": {"reference_answer": "Says transaction records are retained for 7 years (KB-102).",
                 "keywords": ["7 years"], "trajectory": ["search_confluence"]}},
    {"inputs": {"query": "How often are privileged access reviews done?"},
     "outputs": {"reference_answer": "Says privileged access is reviewed quarterly (KB-103).",
                 "keywords": ["quarterly"], "trajectory": ["search_confluence"]}},
    {"inputs": {"query": "What must a vendor that handles customer data do before go-live?"},
     "outputs": {"reference_answer": "Says the vendor must complete the security questionnaire and sign a DPA (KB-104).",
                 "keywords": ["questionnaire"], "trajectory": ["search_confluence"]}},
    {"inputs": {"query": "What is the SLA to acknowledge a card dispute?"},
     "outputs": {"reference_answer": "Says card disputes are acknowledged within 1 business day (KB-105).",
                 "keywords": ["1 business day"], "trajectory": ["search_confluence"]}},
    {"inputs": {"query": "What is the status of OPS-2102?"},
     "outputs": {"reference_answer": "Says OPS-2102 is Blocked.",
                 "keywords": ["Blocked"], "trajectory": ["lookup_jira"]}},
    {"inputs": {"query": "Look at OPS-2102, find what our vendor policy requires, and add it as a comment on the ticket."},
     "outputs": {"reference_answer": "Looks up the ticket, finds KB-104, and adds a comment that mentions the DPA requirement.",
                 "keywords": ["OPS-2102"], "trajectory": ["lookup_jira", "search_confluence", "update_jira"]}},
    {"inputs": {"query": "What is our policy on holding customer crypto assets?"},
     "outputs": {"reference_answer": "Says the knowledge base does not cover this. Does not invent a policy.",
                 "keywords": [], "trajectory": ["search_confluence"]}},
]
print(len(EXAMPLES), "examples")

##  create the dataset and pin version v1

Every change to a dataset creates a new **version**. Tagging one lets any run, now or in CI next month, ask for exactly those examples. The next cell holds this code. It creates the dataset, uploads the examples, and tags the result `v1`. Rerunning is safe:

```python
import time

if client.has_dataset(dataset_name=DATASET):
    dataset = client.read_dataset(dataset_name=DATASET)
    print(f"'{DATASET}' already exists; leaving it as is")
else:
    dataset = client.create_dataset(DATASET)
    client.create_examples(
        inputs=[e["inputs"] for e in EXAMPLES],
        outputs=[e["outputs"] for e in EXAMPLES],
        dataset_id=dataset.id,
    )
    print(f"Created '{DATASET}' with {len(EXAMPLES)} examples")

# Pin this exact set of examples as version "v1", once.
for _ in range(15):
    versions = list(client.list_dataset_versions(dataset_name=DATASET))
    if versions:
        break
    time.sleep(1)
if not any("v1" in (v.tags or []) for v in versions):
    latest = max(versions, key=lambda v: v.as_of)
    client.update_dataset_tag(dataset_name=DATASET, as_of=latest.as_of, tag="v1")
    print("Tagged the current examples as v1")
else:
    print("Version v1 already exists")
```

In [ ]:
# Solution. In the workshop this cell was blank and you wrote it.
import time

if client.has_dataset(dataset_name=DATASET):
    dataset = client.read_dataset(dataset_name=DATASET)
    print(f"'{DATASET}' already exists; leaving it as is")
else:
    dataset = client.create_dataset(DATASET)
    client.create_examples(
        inputs=[e["inputs"] for e in EXAMPLES],
        outputs=[e["outputs"] for e in EXAMPLES],
        dataset_id=dataset.id,
    )
    print(f"Created '{DATASET}' with {len(EXAMPLES)} examples")

# Pin this exact set of examples as version "v1", once.
for _ in range(15):
    versions = list(client.list_dataset_versions(dataset_name=DATASET))
    if versions:
        break
    time.sleep(1)
if not any("v1" in (v.tags or []) for v in versions):
    latest = max(versions, key=lambda v: v.as_of)
    client.update_dataset_tag(dataset_name=DATASET, as_of=latest.as_of, tag="v1")
    print("Tagged the current examples as v1")
else:
    print("Version v1 already exists")

## The target function

A target takes one example's `inputs` and returns whatever the evaluators need: the answer, the tools the agent called, and the tool results it retrieved. Each call uses its own thread, so examples don't affect each other.

In [ ]:
agent = build_agent()


def run_agent(inputs: dict) -> dict:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config={"configurable": {"thread_id": str(uuid7())}},
    )
    return {
        "response": result["messages"][-1].text,
        "trajectory": extract_tool_calls(result["messages"]),
        "sources": extract_sources(result["messages"]),
    }

## An evaluator

An evaluator is a function that returns `{"key": ..., "score": ...}`. This one is deliberately simple: the fraction of expected `keywords` that appear in the answer.

In [ ]:
def keyword_match(outputs, reference_outputs):
    keywords = reference_outputs["keywords"]
    if not keywords:
        return {"key": "keyword_match", "score": 1.0}
    text = outputs["response"].lower()
    return {"key": "keyword_match", "score": sum(k.lower() in text for k in keywords) / len(keywords)}

## Run the experiment on the pinned version

`client.evaluate()` runs the target on every example, applies the evaluators, and records one **experiment** in LangSmith. `as_of="v1"` asks for the tagged version, not whatever the dataset holds today. This takes about a minute.

In [ ]:
examples = client.list_examples(dataset_name=DATASET, as_of="v1")

results = client.evaluate(
    run_agent,
    data=list(examples),
    evaluators=[keyword_match],
    experiment_prefix="first-eval",
    max_concurrency=4,
)
print("experiment:", results.experiment_name)
lab_eval.compare(prefixes=("first-eval",))

**Result:** a table with `first-eval-...`, 8 runs, and an average `keyword_match`. Open **Datasets & Experiments** in LangSmith, open your dataset, and check the **Versions** view: `v1` is there. Then open the experiment and find the lowest-scoring row.

A string check can't tell whether an answer came from your documents or from the model's memory. Challenge 3 adds a check that can.